In [1]:
%%file constant.py

import numpy as np

from acadia.runtime import Runtime
from acadia.data import DataManager
                
class ConstantRuntime(Runtime):
    
    @classmethod
    def main(cls, directory: str, datamanager: DataManager):
        import time
        import numpy as np
        
        from acadia.system import Acadia
        from acadia.arrays import ConstantWaveform
        
        acadia = Acadia()

        pulse_channel = acadia.DAC(4)
        pulse = ConstantWaveform(pulse_channel, length=500e-9)
        
        # Create a sequence for the sequencer
        def sequence(a):
            with a.channel_synchronizer(block=False):
                a.generate(pulse)
                a.generate(pulse)
                a.generate(pulse)
                
            with a.sequencer().loop():
                with a.sequencer().test(a.dma_fifo_occupancy(pulse_channel) == 0):
                    with a.channel_synchronizer(block=False):
                        a.generate(pulse)

        acadia.attach()
        pulse.populate(0.2)

        pulse_channel.set_nyquist_zone(2)
        pulse_channel.configure_nco(frequency=1000e6)
        pulse_channel.set_vop(4000)
        
        acadia.compile(sequence)
        acadia.run(block=False)
        
        while True:
            pass

Overwriting constant.py


In [2]:
import logging
logging.basicConfig(level=logging.DEBUG, 
                    filename="/home/billy/runtime.log", 
                    filemode="w",
                    format='[%(asctime)s] %(levelname)s at %(funcName)s (%(filename)s, %(lineno)d): %(message)s')


from constant import ConstantRuntime
rt = ConstantRuntime()
rt.run("constant.py", "192.168.2.69", display=False)

Button(description='Stop', style=ButtonStyle(), tooltip='Click to stop all local and remote processes.')

'110123-231305'

In [3]:
rt.stop()